# Fixing Links Notebook

Before running this notebook, make sure to run the following command in the terminal to install the required packages:

```bash
bundle install
make all
ruby parse_htmlproofer_log.rb 
```

Each command should be run separately and the final two commands create files for all the htmlproofer errors and warnings. This notebook loads the final csv file to help you see what links exists. You will also need to install the `pandas` library if you haven't already. You can do this by running:

```bash
pip install pandas
```

## Load Libraries and Data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("htmlproofer-report.csv")
# Lower case the column names
df.columns = df.columns.str.lower()
print(f"Number of errors: {len(df)}")

Number of errors: 92


In [3]:
message_counts = df.message.value_counts().reset_index()
print(f"Number of unique error messages: {len(message_counts)}")
message_counts[(message_counts['count']>1)]

Number of unique error messages: 72


,message,count
0,External link https://www.libreoffice.org/get-...,4
1,External link https://central.github.com/mac/l...,4
2,image /images/website/index/woman-using-tabula...,3
3,internally linking to /en/lessons/creating-net...,3
4,image /images/website/individual/red-bird-on-b...,2
5,image /images/website/ipp/bird-with-eggs-nest....,2
6,External link https://www.hastac.org/blogs/har...,2
7,internally linking to /en/lessons/clustering-w...,2
8,internally linking to /en/lessons/text-mining-...,2
9,internally linking to /en/lessons/jupyter-note...,2


In [4]:
file_counts = df.file.value_counts().reset_index()
print(f"Number of unique files with errors: {len(file_counts)}")
file_counts[file_counts['count']>1]

Number of unique files with errors: 57


,file,count
0,_site/en/lessons/retired/intro-to-augmented-re...,4
1,_site/en/lessons/retired/graph-databases-and-S...,4
2,_site/fr/lecons/concevoir-base-donnees-nodegoa...,4
3,_site/en/lessons/retired/getting-started-with-...,3
4,_site/es/lecciones/retirada/introduccion-contr...,3
5,_site/en/lesson-retirement-policy/index.html,3
6,_site/en/lessons/clustering-visualizing-word-e...,3
7,_site/en/lessons/interactive-data-visualizatio...,3
8,_site/es/lecciones/construir-repositorio-de-fu...,3
9,_site/es/lecciones/retirada/sparql-datos-abier...,3


In [5]:
file_counts_df = df.file.value_counts().reset_index()
file_counts_df['count_index'] = file_counts_df.index

file_counts_df

,file,count,count_index
0,_site/en/lessons/retired/intro-to-augmented-re...,4,0
1,_site/en/lessons/retired/graph-databases-and-S...,4,1
2,_site/fr/lecons/concevoir-base-donnees-nodegoa...,4,2
3,_site/en/lessons/retired/getting-started-with-...,3,3
4,_site/es/lecciones/retirada/introduccion-contr...,3,4
5,_site/en/lesson-retirement-policy/index.html,3,5
6,_site/en/lessons/clustering-visualizing-word-e...,3,6
7,_site/en/lessons/interactive-data-visualizatio...,3,7
8,_site/es/lecciones/construir-repositorio-de-fu...,3,8
9,_site/es/lecciones/retirada/sparql-datos-abier...,3,9


In [7]:
merged_df = df.merge(file_counts_df, on='file', how='outer').sort_values(by="count_index", ascending=True)

In [10]:
# import os
# import re

# EXTENSIONS = (".yml")

# def replace_links_preserving_code_blocks(file_path):
#     with open(file_path, "r", encoding="utf-8") as f:
#         content = f.read()

#     # Match code blocks (triple backticks) and inline code (`...`)
#     code_blocks = list(re.finditer(r"(```.*?```|`[^`]*`)", content, re.DOTALL))
#     modified = content
#     offset = 0

#     for match in code_blocks:
#         start, end = match.span()
#         segment = content[start:end]

#         # Temporarily mark this section to skip
#         placeholder = f"%%CODEBLOCK{start}%%"
#         modified = modified[:start + offset] + placeholder + modified[end + offset:]
#         offset += len(placeholder) - (end - start)

#     # Replace all http:// with https://
#     modified = re.sub(r"http://", "https://", modified)

#     # Restore code blocks untouched
#     for match in code_blocks:
#         start = match.start()
#         placeholder = f"%%CODEBLOCK{start}%%"
#         modified = modified.replace(placeholder, match.group(0))

#     if content != modified:
#         print(f"✅ Updated: {file_path}")
#         with open(file_path, "w", encoding="utf-8") as f:
#             f.write(modified)

# def process_all_files(root="."):
#     for dirpath, _, filenames in os.walk(root):
#         for fname in filenames:
#             if fname.endswith(EXTENSIONS) and "ph_authors" in fname:
#                 replace_links_preserving_code_blocks(os.path.join(dirpath, fname))

# process_all_files()